# Statistical and Machine Learning — HW2
## Uniform Convergence and the Limits of Learning

**Student:** Younes Hadli  
**Master 2 — IM / IMDS / MIAGE**  
**Université de Haute-Alsace — Faculté des Sciences et Techniques**

This notebook studies empirical-risk concentration, finite-class uniform convergence, the
No-Free-Lunch phenomenon, and the role of inductive bias. All experiments use direct
Monte-Carlo simulations with NumPy and Matplotlib; no machine-learning library is required.


## Experimental setup

The simulations use a fixed random seed for reproducibility. The Monte-Carlo repetition
counts are large enough to make the curves stable while remaining practical to execute.
The class sizes, domain sizes, sample sizes, accuracy `epsilon`, and confidence level `delta`
are varied rather than tuned to one particular result.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

RNG = np.random.default_rng(2026)
N_REP = 5000
epsilon = 0.10
delta = 0.05
plt.style.use("seaborn-v0_8-whitegrid")

def hoeffding_bound(m, eps=epsilon):
    return np.minimum(1.0, 2 * np.exp(-2 * m * eps**2))

def uniform_bound(m, k, eps=epsilon):
    return np.minimum(1.0, 2 * k * np.exp(-2 * m * eps**2))


## Part I — Simulating uniform convergence

### 1. Empirical risk as a random variable

In [ ]:
p_values = [0.10, 0.30, 0.70]
m_values = [20, 100, 500]
fig, axes = plt.subplots(len(p_values), len(m_values), figsize=(14, 9), constrained_layout=True)

for i, p in enumerate(p_values):
    for j, m in enumerate(m_values):
        empirical_risks = RNG.binomial(m, p, size=N_REP) / m
        axes[i, j].hist(empirical_risks, bins=np.arange(-0.005, 1.006, 0.01),
                        color="steelblue", alpha=0.8, edgecolor="white")
        axes[i, j].axvline(p, color="crimson", linestyle="--", label=f"true risk p={p:.1f}")
        axes[i, j].set_title(f"p={p:.1f}, m={m}")
        axes[i, j].set_xlabel("$L_S(h)$")
        axes[i, j].set_ylabel("count")
        axes[i, j].legend()
        print(f"p={p:.1f}, m={m:3d}: empirical mean={empirical_risks.mean():.4f}, "
              f"bias={empirical_risks.mean()-p:+.4f}, std={empirical_risks.std():.4f}")
plt.show()


**Interpretation.** The empirical mean is close to the fixed true risk because
`E[L_S(h)] = L_D(h) = p`. Increasing `m` makes the histogram narrower: for a binomial
sample, the variance of the empirical risk is `p(1-p)/m`, so the typical fluctuation
decreases at the expected `1/sqrt(m)` rate.


### 2. Deviation between empirical and true risk

In [ ]:
p = 0.30
m_grid = np.array([10, 20, 40, 80, 160, 320, 640])
empirical_prob = []
for m in m_grid:
    deviations = np.abs(RNG.binomial(m, p, size=N_REP) / m - p)
    empirical_prob.append(np.mean(deviations > epsilon))
empirical_prob = np.array(empirical_prob)
bound = hoeffding_bound(m_grid)

plt.figure(figsize=(8, 5))
plt.plot(m_grid, empirical_prob, "o-", label="Monte-Carlo probability")
plt.plot(m_grid, bound, "s--", label="Hoeffding bound")
plt.xscale("log", base=2)
plt.ylim(-0.02, 1.05)
plt.xlabel("sample size m")
plt.ylabel(r"$P(|L_S-L_D|>\epsilon)$")
plt.title(f"One hypothesis: p={p}, epsilon={epsilon}")
plt.legend()
plt.show()
print("m       empirical       Hoeffding")
for m, observed, theoretical in zip(m_grid, empirical_prob, bound):
    print(f"{m:4d}    {observed:9.4f}      {theoretical:9.4f}")


**Interpretation.** The experimental probability decreases with `m`, as predicted.
The Hoeffding curve is an upper bound, not an equality: it is distribution-free and therefore
usually conservative, especially when the actual binomial variance is smaller than the
worst-case variance used by the bound.


### 3. From one hypothesis to uniform convergence

In [ ]:
K = 50
p_hypotheses = np.linspace(0.05, 0.95, K)
uniform_prob = []
for m in m_grid:
    empirical = RNG.binomial(m, p_hypotheses, size=(N_REP, K)) / m
    delta_s = np.max(np.abs(empirical - p_hypotheses), axis=1)
    uniform_prob.append(np.mean(delta_s > epsilon))
uniform_prob = np.array(uniform_prob)
class_bound = uniform_bound(m_grid, K)

plt.figure(figsize=(8, 5))
plt.plot(m_grid, uniform_prob, "o-", label=f"Monte Carlo, |H|={K}")
plt.plot(m_grid, class_bound, "s--", label=r"$2|H|e^{-2m\epsilon^2}$")
plt.xscale("log", base=2)
plt.ylim(-0.02, 1.05)
plt.xlabel("sample size m")
plt.ylabel(r"$P(\Delta_S>\epsilon)$")
plt.title("Finite-class uniform convergence")
plt.legend()
plt.show()


**Interpretation.** Uniform convergence controls the largest deviation over all
hypotheses, so it is a stronger event than concentration for one fixed hypothesis. The union
bound explains the factor `|H|`: any one of the hypotheses may be the one that deviates.
The finite-class bound is again sufficient and can be noticeably above the measured probability.


### 4. Effect of the size of the hypothesis class

In [ ]:
class_sizes = [5, 10, 50, 100, 500]
class_results = {}
for k in class_sizes:
    risks = np.linspace(0.05, 0.95, k)
    values = []
    for m in m_grid:
        empirical = RNG.binomial(m, risks, size=(N_REP, k)) / m
        values.append(np.mean(np.max(np.abs(empirical-risks), axis=1) > epsilon))
    class_results[k] = np.array(values)

plt.figure(figsize=(9, 5))
for k in class_sizes:
    plt.plot(m_grid, class_results[k], "o-", label=f"|H|={k}")
plt.xscale("log", base=2)
plt.ylim(-0.02, 1.05)
plt.xlabel("sample size m")
plt.ylabel(r"$P(\Delta_S>\epsilon)$")
plt.title("Uniform convergence for different class sizes")
plt.legend(ncol=2)
plt.show()


**Interpretation.** At a fixed sample size, larger classes have more opportunities
for an unusually large deviation, so their uniform-convergence probability is higher. More
data are consequently needed. The theoretical sample complexity grows as `log(|H|)`, not
linearly: increasing the class can be costly, but the cost is logarithmic in this finite-class
bound.


### 5. Experimental sample complexity

In [ ]:
target_k = 50
sample_grid = np.arange(20, 1001, 20)
probabilities = []
risks = np.linspace(0.05, 0.95, target_k)
for m in sample_grid:
    empirical = RNG.binomial(m, risks, size=(N_REP, target_k)) / m
    probabilities.append(np.mean(np.max(np.abs(empirical-risks), axis=1) > epsilon))
probabilities = np.array(probabilities)
eligible = sample_grid[probabilities <= delta]
experimental_m = int(eligible[0]) if len(eligible) else None
theoretical_m = int(np.ceil(np.log(2 * target_k / delta) / (2 * epsilon**2)))

plt.figure(figsize=(8, 5))
plt.plot(sample_grid, probabilities, label="Monte Carlo")
plt.axhline(delta, color="crimson", linestyle="--", label=f"delta={delta}")
plt.axvline(theoretical_m, color="darkgreen", linestyle=":", label=f"bound m={theoretical_m}")
if experimental_m is not None:
    plt.axvline(experimental_m, color="purple", linestyle="-.", label=f"first tested m={experimental_m}")
plt.xlabel("sample size m")
plt.ylabel(r"$P(\Delta_S>\epsilon)$")
plt.title(f"Sample complexity, |H|={target_k}, epsilon={epsilon}")
plt.legend()
plt.show()
print(f"Experimental first tested m with probability <= delta: {experimental_m}")
print(f"Theoretical sufficient m from the union/Hoeffding bound: {theoretical_m}")


**Interpretation.** The theoretical value is a sufficient guarantee obtained from
an upper bound and is not expected to be the exact smallest value. The experimental value also
depends on the tested grid and on Monte-Carlo noise; the guarantee is conservative by design.


## Part II — Simulating No-Free-Lunch

### 6. Seen and unseen inputs

In [ ]:
def unseen_fraction_once(N, m):
    sampled_inputs = RNG.integers(0, N, size=m)
    return np.mean(~np.isin(np.arange(N), sampled_inputs))

domain_settings = [(50, 25), (50, 100), (200, 100), (200, 500)]
observed_unseen = []
theoretical_unseen = []
for N, m in domain_settings:
    values = np.array([unseen_fraction_once(N, m) for _ in range(N_REP)])
    observed_unseen.append(values.mean())
    theoretical_unseen.append((1 - 1/N)**m)
    print(f"N={N:3d}, m={m:3d}: observed={values.mean():.4f}, "
          f"theory={(1-1/N)**m:.4f}")

labels = [f"N={N}, m={m}" for N, m in domain_settings]
x = np.arange(len(labels))
plt.figure(figsize=(9, 5))
plt.bar(x - 0.18, observed_unseen, width=0.36, label="Monte Carlo")
plt.bar(x + 0.18, theoretical_unseen, width=0.36, label=r"$(1-1/N)^m$")
plt.xticks(x, labels)
plt.ylabel("fraction of unseen inputs")
plt.title("Expected fraction of unseen domain points")
plt.legend()
plt.show()


**Interpretation.** For a fixed point `x`, every draw misses it with probability
`1-1/N`; independence across the `m` draws gives `(1-1/N)^m`. By linearity of expectation,
the expected fraction of unseen points is the same probability, even though the indicators
for different points are not independent.


### 7. Error on unseen inputs

In [ ]:
N = 100
risk_m_grid = np.array([0, 10, 25, 50, 100, 200, 400, 800])
observed_risk = []
for m in risk_m_grid:
    risks = []
    for _ in range(N_REP):
        target = RNG.integers(0, 2, size=N)
        sampled = RNG.integers(0, N, size=m)
        predictions = np.zeros(N, dtype=int)
        if m:
            predictions[sampled] = target[sampled]
        risks.append(np.mean(predictions != target))
    observed_risk.append(np.mean(risks))
observed_risk = np.array(observed_risk)
theoretical_risk = 0.5 * (1 - 1/N)**risk_m_grid

plt.figure(figsize=(8, 5))
plt.plot(risk_m_grid, observed_risk, "o-", label="Monte Carlo risk")
plt.plot(risk_m_grid, theoretical_risk, "s--", label=r"$rac12(1-1/N)^m$")
plt.xlabel("sample size m")
plt.ylabel("expected true risk")
plt.title(f"Memorization learner on random targets (N={N})")
plt.legend()
plt.show()
print("m       observed risk   theoretical risk")
for m, observed, theoretical in zip(risk_m_grid, observed_risk, theoretical_risk):
    print(f"{m:4d}      {observed:.4f}          {theoretical:.4f}")


**Interpretation.** Seen labels are memorized and therefore correct. On unseen
points, the target bit is independent of the learner's default prediction, so the error rate
is one half. Multiplying this by the expected unseen fraction yields the displayed formula.


### 8. Different predictions on unseen inputs

In [ ]:
strategies = ["always 0", "always 1", "random Bernoulli(1/2)"]
strategy_risks = {name: [] for name in strategies}
for m in [0, 20, 100, 300]:
    totals = {name: [] for name in strategies}
    for _ in range(N_REP):
        target = RNG.integers(0, 2, size=N)
        sampled = RNG.integers(0, N, size=m)
        seen = np.zeros(N, dtype=bool)
        seen[sampled] = True
        memorized = np.zeros(N, dtype=int)
        if m:
            memorized[sampled] = target[sampled]
        for name in strategies:
            predictions = memorized.copy()
            unseen = ~seen
            if name == "always 1":
                predictions[unseen] = 1
            elif name == "random Bernoulli(1/2)":
                predictions[unseen] = RNG.integers(0, 2, size=unseen.sum())
            totals[name].append(np.mean(predictions != target))
    for name in strategies:
        strategy_risks[name].append(np.mean(totals[name]))

plt.figure(figsize=(8, 5))
for name in strategies:
    plt.plot([0, 20, 100, 300], strategy_risks[name], "o-", label=name)
plt.xlabel("sample size m")
plt.ylabel("expected true risk")
plt.title("Unseen-point rules on completely random targets")
plt.legend()
plt.show()
for name in strategies:
    print(name, np.round(strategy_risks[name], 4))


**Interpretation.** The three rules have essentially the same expected risk:
on an unseen point, a fixed prediction and an independent random prediction each disagree
with a random target with probability `1/2`. None consistently dominates another. This is
the No-Free-Lunch message: without assumptions about target functions, the sample contains
no information about the labels of unseen points.


## Part III — Introducing prior knowledge

### 9. Learning with a matching inductive bias

In [ ]:
def threshold_labels(N, theta):
    x = np.arange(1, N + 1)
    return (x > theta).astype(int)

def erm_threshold(N, sample_x, sample_y):
    candidates = np.arange(1, N)
    losses = []
    for t in candidates:
        predictions = (sample_x > t).astype(int)
        losses.append(np.mean(predictions != sample_y))
    return int(candidates[np.argmin(losses)])

N_threshold = 100
threshold_m_grid = np.array([1, 2, 5, 10, 20, 40, 80, 160, 320])
threshold_risk = []
for m in threshold_m_grid:
    risks = []
    for _ in range(N_REP):
        theta = int(RNG.integers(1, N_threshold))
        target = threshold_labels(N_threshold, theta)
        sample_x = RNG.integers(1, N_threshold + 1, size=m)
        sample_y = target[sample_x - 1]
        learned_t = erm_threshold(N_threshold, sample_x, sample_y)
        learned = threshold_labels(N_threshold, learned_t)
        risks.append(np.mean(learned != target))
    threshold_risk.append(np.mean(risks))
threshold_risk = np.array(threshold_risk)

plt.figure(figsize=(8, 5))
plt.plot(threshold_m_grid, threshold_risk, "o-", label="threshold ERM")
plt.xscale("log", base=2)
plt.xlabel("sample size m")
plt.ylabel("expected true risk")
plt.title("ERM with the correct threshold inductive bias")
plt.legend()
plt.show()
print("m       expected risk")
for m, risk in zip(threshold_m_grid, threshold_risk):
    print(f"{m:4d}      {risk:.4f}")


**Interpretation.** Although the target is unknown outside the sample, all targets
belong to the same threshold family. ERM therefore only needs to locate the transition point;
it does not have to guess an independent label for every unseen input. The risk decreases
much faster than in the arbitrary-label experiment.


### 10. Random targets versus structured targets

In [ ]:
random_target_risk = 0.5 * (1 - 1/N_threshold)**threshold_m_grid
plt.figure(figsize=(8, 5))
plt.plot(threshold_m_grid, random_target_risk, "o--",
         label="random targets + memorization (theory)")
plt.plot(threshold_m_grid, threshold_risk, "s-",
         label="threshold targets + threshold ERM (Monte Carlo)")
plt.xscale("log", base=2)
plt.xlabel("sample size m")
plt.ylabel("expected true risk")
plt.title("Generalization with and without structural assumptions")
plt.legend()
plt.show()
print(f"At m={threshold_m_grid[-1]}, random-target risk={random_target_risk[-1]:.4f}, "
      f"threshold-ERM risk={threshold_risk[-1]:.4f}")


**Interpretation.** The threshold learner transfers information between nearby
unseen inputs through the shared boundary parameter. This is an inductive bias: the learner
assumes a restricted target family, and that assumption is exactly correct in this experiment.
For completely random targets there is no such transferable structure, so memorization cannot
predict unseen labels better than chance.


## 11. Final analysis and conclusion

* `L_S(h)` is an average of independent Bernoulli errors. Its expectation equals `L_D(h)`,
  while its variance decreases as `1/m`, so empirical risk concentrates around true risk.
* Uniform convergence is a simultaneous statement over every hypothesis. A union bound
  introduces the factor `|H|`, and larger classes require more data; the resulting dependence
  is logarithmic in `|H|` in the finite-class sample-complexity expression.
* Hoeffding and union bounds are safe distribution-free guarantees, not exact predictions.
  Their conservatism explains why the theoretical sufficient sample size can exceed the
  smallest value observed experimentally.
* With completely random target functions, labels of unseen inputs are independent of the
  training sample. Any fixed, memorized, or random default rule therefore has expected
  unseen-point error `1/2`, which illustrates No-Free-Lunch.
* Restricting both target and hypothesis classes to thresholds introduces prior knowledge.
  The matching inductive bias lets ERM infer one boundary and generalize to many unseen
  points, producing much smaller risk than on arbitrary targets.

Overall, the experiments show that convergence alone does not guarantee useful learning:
generalization also depends on controlling the hypothesis class and on having structural
assumptions that connect observed and unobserved examples.
